In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
import os

from datetime import datetime

# Ensure src is in the path
import sys

sys.path.insert(0, os.path.abspath("../src"))

from src.config import Config
from src.utils import set_seed, get_device, to_tensor
from src.data import SimpleGridWorld
from src.agents import (
    DQNAgent,
    DynaQAgent,
    HierarchicalActorCritic,
    GoalConditionedAgent,
    ModelPredictiveController,
    MonteCarloTreeSearch,
)
from src.train import (
    train_agent,
    run_model_based_experiments,
    run_hierarchical_experiments,
    run_planning_experiments,
)


# 1. Configuration and Setup
# Initialize configuration
config = Config()
set_seed(config.general.seed)
device = get_device()

print(f"Using device: {device}")
print(f"Results will be saved to: {config.general.results_dir}")
print(f"Pictures will be saved to: {config.general.pictures_dir}")

# Create directories if they don't exist
os.makedirs(config.general.results_dir, exist_ok=True)
os.makedirs(config.general.pictures_dir, exist_ok=True)

## 2. Environment Initialization

We will use the `SimpleGridWorld` environment for demonstration. You can switch to other Gymnasium environments by modifying `config.environment.env_name` and updating the relevant dimensions in `config.py`.


In [ ]:
# Initialize environment
env = SimpleGridWorld(grid_size=config.environment.grid_size)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(f"Environment: {config.environment.env_name}")
print(f"State Dimension: {state_dim}")
print(f"Action Dimension: {action_dim}")

# Update config with environment dimensions
config.dynamics_model.state_dim = state_dim
config.dynamics_model.action_dim = action_dim
config.manager.state_dim = state_dim
config.manager.subgoal_dim = (
    state_dim  # Assuming subgoals are states for SimpleGridWorld
)
config.worker.state_dim = state_dim
config.worker.action_dim = action_dim
config.worker.goal_dim = state_dim

## 3. Data Visualization (Example)

Here, we can add code to visualize the environment, agent trajectories, or raw data if applicable.


In [ ]:
# Example: Visualize a random trajectory in SimpleGridWorld
def plot_gridworld_trajectory(
    env_instance: SimpleGridWorld,
    agent_path: List[np.ndarray],
    title: str,
    filename: str,
):
    fig, ax = plt.subplots(figsize=(6, 6))
    grid_size = env_instance.grid_size
    goal_pos = env_instance.goal_pos

    # Draw grid lines
    for x in range(grid_size + 1):
        ax.axvline(x, color="lightgray", lw=0.5)
        ax.axhline(x, color="lightgray", lw=0.5)

    # Plot goal
    ax.plot(
        goal_pos[0] + 0.5,
        goal_pos[1] + 0.5,
        "P",
        markersize=15,
        color="green",
        label="Goal",
    )

    # Plot agent path
    path_x = [p[0] + 0.5 for p in agent_path]
    path_y = [p[1] + 0.5 for p in agent_path]
    ax.plot(
        path_x,
        path_y,
        "-o",
        color="blue",
        markersize=5,
        linewidth=2,
        label="Agent Path",
    )
    ax.plot(path_x[0], path_y[0], "o", markersize=8, color="red", label="Start")

    ax.set_xlim(0, grid_size)
    ax.set_ylim(0, grid_size)
    ax.set_xticks(np.arange(0.5, grid_size + 0.5), labels=np.arange(grid_size))
    ax.set_yticks(np.arange(0.5, grid_size + 0.5), labels=np.arange(grid_size))
    ax.set_xlabel("X-coordinate")
    ax.set_ylabel("Y-coordinate")
    ax.set_title(title)
    ax.legend()
    ax.set_aspect("equal", adjustable="box")
    plt.gca().invert_yaxis()  # Invert y-axis to match typical grid conventions (0,0 top-left)
    plt.savefig(os.path.join(config.general.pictures_dir, filename), dpi=300)
    plt.show()


# Generate a random path for demonstration
random_env = SimpleGridWorld(grid_size=config.environment.grid_size)
random_path = []
state, _ = random_env.reset()
random_path.append(state)
done = False
for _ in range(20):
    action = random_env.action_space.sample()
    state, reward, done, truncated, _ = random_env.step(action)
    random_path.append(state)
    if done or truncated:
        break

plot_gridworld_trajectory(
    random_env,
    random_path,
    "Random Walk in SimpleGridWorld",
    "gridworld_random_walk.png",
)

## 4. Training Loop

We will run the main training experiments here, utilizing the `run_model_based_experiments`, `run_hierarchical_experiments`, and `run_planning_experiments` functions from `src/train.py`.


In [ ]:
# Run all experiments
all_experiment_results = {}

print("\n--- Running Model-Based Experiments ---")
model_based_results = run_model_based_experiments(config)
all_experiment_results["model_based"] = model_based_results

print("\n--- Running Hierarchical Experiments ---")
hierarchical_results = run_hierarchical_experiments(config)
all_experiment_results["hierarchical"] = hierarchical_results

print("\n--- Running Planning Experiments ---")
planning_results = run_planning_experiments(config)
all_experiment_results["planning"] = planning_results

print("\nAll experiments completed.")
print(all_experiment_results)

## 5. Visualization of Results

Here we will generate publication-quality figures to analyze the performance of the trained agents, including learning curves, loss plots, and specific agent behaviors.


In [ ]:
def create_comprehensive_visualizations(
    results_dict: Dict[str, Any],
    config: Config,
    title: str = "Comprehensive RL Analysis",
):
    """Create comprehensive visualizations from experiment results."""
    plt.style.use("seaborn-v0_8-darkgrid")
    sns.set_palette("viridis")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(title, fontsize=18, fontweight="bold")

    # Plot 1: Episode Rewards Over Time for Model-Based RL
    ax1 = axes[0, 0]
    if "model_based" in results_dict and "dyna_q" in results_dict["model_based"]:
        rewards = results_dict["model_based"]["dyna_q"]["episode_rewards"]
        episodes = np.arange(len(rewards))
        ax1.plot(episodes, rewards, label="Dyna-Q Agent", color="blue", alpha=0.7)
        ax1.set_title("Dyna-Q Agent Episode Rewards")
        ax1.set_xlabel("Episode")
        ax1.set_ylabel("Total Reward")
        ax1.legend()
    else:
        ax1.set_title("Dyna-Q Agent Episode Rewards (No Data)")

    # Plot 2: Episode Rewards Over Time for Hierarchical RL
    ax2 = axes[0, 1]
    if (
        "hierarchical" in results_dict
        and "goal_conditioned" in results_dict["hierarchical"]
    ):
        gc_rewards = results_dict["hierarchical"]["goal_conditioned"]["episode_rewards"]
        episodes = np.arange(len(gc_rewards))
        ax2.plot(
            episodes,
            gc_rewards,
            label="Goal-Conditioned Agent",
            color="green",
            alpha=0.7,
        )
        if "hierarchical_ac" in results_dict["hierarchical"]:
            hac_rewards = results_dict["hierarchical"]["hierarchical_ac"][
                "episode_rewards"
            ]
            ax2.plot(
                np.arange(len(hac_rewards)),
                hac_rewards,
                label="Hierarchical AC Agent",
                color="orange",
                alpha=0.7,
            )
        ax2.set_title("Hierarchical RL Episode Rewards")
        ax2.set_xlabel("Episode")
        ax2.set_ylabel("Total Reward")
        ax2.legend()
    else:
        ax2.set_title("Hierarchical RL Episode Rewards (No Data)")

    # Plot 3: Loss Curves (Example: Q-Loss for Dyna-Q and Critic Loss for HAC)
    ax3 = axes[1, 0]
    if (
        "model_based" in results_dict
        and "dyna_q" in results_dict["model_based"]
        and results_dict["model_based"]["dyna_q"]["q_losses"]
    ):
        q_losses = results_dict["model_based"]["dyna_q"]["q_losses"]
        ax3.plot(
            np.arange(len(q_losses)),
            q_losses,
            label="Dyna-Q Q-Loss",
            color="red",
            alpha=0.7,
        )
    if (
        "hierarchical" in results_dict
        and "hierarchical_ac" in results_dict["hierarchical"]
        and results_dict["hierarchical"]["hierarchical_ac"]["q_losses"]
    ):
        hac_critic_losses = results_dict["hierarchical"]["hierarchical_ac"]["q_losses"]
        ax3.plot(
            np.arange(len(hac_critic_losses)),
            hac_critic_losses,
            label="HAC Critic Loss",
            color="purple",
            alpha=0.7,
        )
    ax3.set_title("Training Loss Curves")
    ax3.set_xlabel("Training Step (Episode)")
    ax3.set_ylabel("Loss")
    ax3.legend()

    # Plot 4: Planning Algorithms Performance
    ax4 = axes[1, 1]
    if "planning" in results_dict and "mpc" in results_dict["planning"]:
        mpc_rewards = results_dict["planning"]["mpc"]["episode_rewards"]
        ax4.plot(
            np.arange(len(mpc_rewards)),
            mpc_rewards,
            label="MPC Agent",
            color="brown",
            alpha=0.7,
        )
    if "planning" in results_dict and "mcts" in results_dict["planning"]:
        mcts_rewards = results_dict["planning"]["mcts"]["episode_rewards"]
        ax4.plot(
            np.arange(len(mcts_rewards)),
            mcts_rewards,
            label="MCTS Agent",
            color="teal",
            alpha=0.7,
        )
    ax4.set_title("Planning Algorithms Episode Rewards")
    ax4.set_xlabel("Episode")
    ax4.set_ylabel("Total Reward")
    ax4.legend()

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to prevent title overlap
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"comprehensive_analysis_{timestamp}.png"
    plt.savefig(os.path.join(config.general.pictures_dir, filename), dpi=300)
    plt.show()
    print(
        f"Comprehensive analysis saved to: {os.path.join(config.general.pictures_dir, filename)}"
    )


create_comprehensive_visualizations(
    all_experiment_results, config, title="MB-HRL Comprehensive Analysis"
)